# DPO-3: SFT QLoRA Baseline

Reproducing the Zephyr-7B-β SFT step on `mistralai/Mistral-7B-v0.1` using QLoRA.

**Flow:**
1. Load base model → run predictions
2. SFT QLoRA on UltraChat 200K (1 epoch, Zephyr config)
3. Run same predictions → compare

Config source: `D:/git/alignment-handbook/recipes/zephyr-7b-beta/sft/config_qlora.yaml`

In [ ]:
# !pip install transformers peft trl bitsandbytes datasets accelerate -q

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset

MODEL_ID = "mistralai/Mistral-7B-v0.1"
OUTPUT_DIR = "./sft-zephyr-lora"

# Set to an integer (e.g. 2000) for a quick smoke test; None = full dataset
MAX_TRAIN_SAMPLES = None

# Zephyr chat template — from alignment-handbook sft/config_qlora.yaml
CHAT_TEMPLATE = (
    "{% for message in messages %}\n"
    "{% if message['role'] == 'user' %}\n"
    "{{ '<|user|>\\n' + message['content'] + eos_token }}\n"
    "{% elif message['role'] == 'system' %}\n"
    "{{ '<|system|>\\n' + message['content'] + eos_token }}\n"
    "{% elif message['role'] == 'assistant' %}\n"
    "{{ '<|assistant|>\\n'  + message['content'] + eos_token }}\n"
    "{% endif %}\n"
    "{% if loop.last and add_generation_prompt %}\n"
    "{{ '<|assistant|>' }}\n"
    "{% endif %}\n"
    "{% endfor %}"
)

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

## 0. Load Model

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
tokenizer.chat_template = CHAT_TEMPLATE

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    attn_implementation="flash_attention_2",
)
model.config.use_cache = False

print(f"Params: {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B")
print(f"VRAM:   {torch.cuda.memory_allocated() / 1e9:.2f} GB")

## 1. Base Model — Predictions Before SFT

Mistral-7B-v0.1 is a raw completion model with no instruction tuning.
We apply the Zephyr chat template so the prompt format is consistent pre- and post-SFT.

In [ ]:
TEST_PROMPTS = [
    [{"role": "user", "content": "Explain the difference between supervised learning and reinforcement learning in 3 sentences."}],
    [{"role": "user", "content": "Write a short haiku about gradient descent."}],
    [{"role": "user", "content": "What is 17 × 23? Show your work step by step."}],
]

@torch.inference_mode()
def generate(messages, max_new_tokens=256):
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    prompt_len = inputs["input_ids"].shape[1]
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.decode(output_ids[0][prompt_len:], skip_special_tokens=True)

In [ ]:
print("=" * 60)
print("BASE MODEL")
print("=" * 60)

base_outputs = []
for msgs in TEST_PROMPTS:
    out = generate(msgs)
    base_outputs.append(out)
    print(f"\nQ: {msgs[0]['content']}")
    print(f"A: {out}")
    print("-" * 60)

## 2. SFT QLoRA Training

Exact config from `alignment-handbook/recipes/zephyr-7b-beta/sft/config_qlora.yaml`:
- LoRA: r=16, alpha=16, 7 target modules
- lr=2e-4, cosine, warmup 10%, batch=4, grad_accum=2
- 1 epoch, max_seq_length=2048, packing=True

In [ ]:
from datasets import Dataset
from tqdm.auto import tqdm

raw_train = load_dataset("HuggingFaceH4/ultrachat_200k", split="train_sft")
raw_eval  = load_dataset("HuggingFaceH4/ultrachat_200k", split="test_sft")

if MAX_TRAIN_SAMPLES:
    raw_train = raw_train.select(range(MAX_TRAIN_SAMPLES))

def fmt(examples):
    return [
        tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
        for msgs in examples
    ]

train_texts = fmt(raw_train["messages"])
eval_texts  = fmt(raw_eval.select(range(1000))["messages"])  # 1000 held-out examples, matching handbook

ds_train = Dataset.from_dict({"text": train_texts})
ds_eval  = Dataset.from_dict({"text": eval_texts})

print(f"Train: {len(ds_train):,}  |  Eval: {len(ds_eval):,}")
print("\nSample (first 400 chars):")
print(ds_train[0]["text"][:400], "...")

In [ ]:
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=2600,
    bf16=True,
    do_eval=True,
    eval_strategy="epoch",
    logging_steps=5,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=1,
    report_to="none",
    max_length=2048,
    dataset_text_field="text",
    packing=True,    # flash_attention_2 handles sequence boundaries correctly — no cross-contamination
    seed=42,
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=ds_train,
    eval_dataset=ds_eval,
    processing_class=tokenizer,
)

trainer.train()

## 3. Post-SFT Predictions

Same prompts, same `generate()` — only the model weights changed.

In [ ]:
model.eval()

print("=" * 60)
print("POST-SFT MODEL")
print("=" * 60)

sft_outputs = []
for msgs in TEST_PROMPTS:
    out = generate(msgs)
    sft_outputs.append(out)
    print(f"\nQ: {msgs[0]['content']}")
    print(f"A: {out}")
    print("-" * 60)

## 4. Side-by-Side Comparison

In [ ]:
for i, msgs in enumerate(TEST_PROMPTS):
    q = msgs[0]["content"]
    print(f"{'=' * 60}")
    print(f"Q: {q}")
    print(f"\n[BASE]\n{base_outputs[i]}")
    print(f"\n[SFT] \n{sft_outputs[i]}")
    print()